In [ ]:
import pandas as pd
import re

In [ ]:
input_file_path = '/articles_dataset_big.csv'
data = pd.read_csv(input_file_path, index_col=[0])

In [ ]:
data

,id,source_domain,url,title,text,label_3,label_5
0,887d66cad21d7daa12633e6aa18250a7a34ffa4913cf41...,www.mmnews.de,https://www.mmnews.de/politik/28379-bericht-sc...,Schulz: Gratisflüge mit EU-Jets für's Parteive...,Schulz: Gratisflüge mit EU-Jets für's Parteive...,1,center-right
1,7af76faf980452e667200c8e0427bdc5a241c5a9fe73c8...,www.bild.de,https://www.bild.de/politik/inland/politik-inl...,Sachsen hat gewählt: Der Krimi um die leeren A...,Von: Sören Haberlandt und Michael Kruse\nIrrer...,1,center-right
2,444593843e2d0892f5ac204bcfcfafbaf72d82c4e565bf...,www.fr.de,https://www.fr.de/politik/boris-johnson-nicola...,SNP-Chefin greift Boris Johnson an - und forde...,SNP-Chefin greift Boris Johnson an - und forde...,0,center-left
3,4bc150c2bd195ee0edd66c419ca547d2696ae95891bf72...,www.mmnews.de,https://www.mmnews.de/politik/101548-siemens-c...,Siemens-Chef Kaeser will nach Saudi-Arabien,Siemens-Chef Kaeser will nach Saudi-Arabien\nA...,1,center-right
4,9572abfa63f0fec3fc9e9ed216474dcdd87e40447d239c...,www.focus.de,https://www.focus.de/politik/deutschland/theme...,"""Maybrit Illner"": Kanzleramtschef Altmaier ver...",Die von der SPD lange abgelehnte Große Koaliti...,1,center-right
...,...,...,...,...,...,...,...
13850,41d0aff870eb3a8c9609a6c474a0e6591dfe89a6500288...,www.n-tv.de,https://www.n-tv.de/politik/Papst-bedauert-Hag...,"""Es schmerzt mich sehr"": Papst bedauert Hagia-...","Die Entscheidung der Türkei, die Hagia Sophia ...",1,center
13851,40b4e05ee74eb43c09bba6cc69ed8ee0bde78a8d312210...,www.mmnews.de,https://www.mmnews.de/politik/145047-oesterrei...,Österreichs Kanzler: EU darf keine Schulden-Un...,Österreichs Kanzler: EU darf keine Schulden-Un...,1,center-right
13852,d9eec36ed123d5eb5b3b56ed466e97f4d8fb1b50710cd1...,www.bild.de,https://www.bild.de/politik/inland/bundestagsw...,Der Partei fehlen Themen - Panik-Pressekonfere...,"Von: Von FLORIAN KAIN\nDie Flaute in Umfragen,...",1,center-right
13853,196185390051889797bcbed15b632d32932e1a4c3250c3...,www.tichyseinblick.de,https://www.tichyseinblick.de/meinungen/spd-pa...,SPD-Parteitag: Andrea Nahles fügt sich den Rea...,Im Beisein der Vorsitzenden des Deutschen Gewe...,2,far-right


In [ ]:
def clean_dataset(df, output_file_path):

    # Create a copy of the original text column
    df['original_text'] = df['text']

    # Counter for tracking changes
    changes_count = {
        'www.bild.de': {'rule_1_1': 0, 'rule_1_2': 0},
        'www.fr.de': {'rule_2_1': 0, 'rule_2_2': 0},
        'www.focus.de': {'rule_3_1': 0, 'rule_3_2': 0, 'rule_3_3': 0},
        'Symbol': {'rule': 0},
        'Reference': {'short_slash_lines': 0},
        'Time': {'time': 0}
    }

    # Track rows that will be removed due to empty text
    removed_indices = []

    # First check for and track rows with initially empty, None, or NaN text
    for index, row in df.iterrows():
        text = row['text']
        if pd.isna(text) or (isinstance(text, str) and not text.strip()):
            removed_indices.append(index)

    print(f"{removed_indices} rows with initially empty/None/NaN text")
    """
    time_phrase = re.compile(
        r'(Update|News|Die Meldung|Meldung|Ursprungsartikel|Ursprungsmeldung)'
        r'\s*(?:vom|von|,|um)?\s*'
        r'(?:'
            r'(?:\d{1,2}[./]\d{1,2}(?:[./]\d{4})?|'               # a. Fully numeric date e.g., 24.01.2020
            r'(?:\d+\s*[.,]?\s*[A-Za-zäöüÄÖÜß]+(?:\s*\d{4})?))?'  # b. Day + Month + optional year
            r'(?:\s*(?:,|um)?\s*\d{1,2}[.:]\d{2}\s*(?:Uhr)?)?'    # c. Optional time
        r'|\d{1,2}\.\d{1,2}\s*(?:Uhr)?'                           # d. Time without date, optional 'Uhr'
        r')'
        r':'
        , re.IGNORECASE
    )
    """
    time_phrase = r'\b(Update|News|Die Meldung|Meldung|Ursprungsartikel|Ursprungsmeldung)[^:]*:'



    # Process each row in the dataset
    for index, row in df.iterrows():
        # Skip rows already marked for removal
        if index in removed_indices:
            continue

        source_domain = row['source_domain']
        text = row['text']

        # Split the text into lines for processing
        lines = text.split('\n')

        if '►' in ''.join(lines):
            lines = [line.replace('►', '') for line in lines]
            changes_count['Symbol']['rule'] += 1

        if '▶︎' in ''.join(lines):
            lines = [line.replace('▶︎', '') for line in lines]
            changes_count['Symbol']['rule'] += 1

        if '➤' in ''.join(lines):
            lines = [line.replace('➤', '') for line in lines]
            changes_count['Symbol']['rule'] += 1

        if '+++' in ''.join(lines):
            lines = [line.replace('+++', ' ') for line in lines]
            changes_count['Symbol']['rule'] += 1

        lines = [line for line in lines if '©' not in line]
        changes_count['Symbol']['rule'] += 1

        lines = [line for line in lines if '>>>' not in line]
        changes_count['Symbol']['rule'] += 1

        # Remove lines containing "/" that are shorter than 50 characters
        line_count_before = len(lines)
        lines = [line for line in lines if not ('/' in line and len(line) < 50)]
        lines_removed = line_count_before - len(lines)
        if lines_removed > 0:
            changes_count['Reference']['short_slash_lines'] += lines_removed

        # Process each line to remove time phrases
        time_phrases_found = 0
        for i in range(len(lines)):
            # Count matches before replacement
            matches = re.findall(time_phrase, lines[i])
            time_phrases_found += len(matches)

            # Remove the time phrases
            lines[i] = re.sub(time_phrase, '', lines[i])

        if time_phrases_found > 0:
            changes_count['Time']['time'] += time_phrases_found

        # Apply rules based on source_domain
        if source_domain == 'www.bild.de':
            # Rule 1.1: Remove lines starting with "Von: " in first or second position
            if len(lines) > 0 and lines[0].startswith('Von: '):
                lines.pop(0)
                changes_count['www.bild.de']['rule_1_1'] += 1
            elif len(lines) > 1 and lines[1].startswith('Von: '):
                lines.pop(1)
                changes_count['www.bild.de']['rule_1_1'] += 1

            # Rule 1.2: Remove PS line at the end
            if lines and lines[-1].strip() == "PS: Sind Sie bei Facebook? Werden Sie Fan von BILD.de-Politik!":
                lines.pop()
                changes_count['www.bild.de']['rule_1_2'] += 1

        elif source_domain == 'www.fr.de':
            # Rule 2.1: Remove second line if it starts with "Von: "
            if len(lines) > 1 and lines[1].startswith('Von: '):
                lines.pop(1)
                changes_count['www.fr.de']['rule_2_1'] += 1

            # Rule 2.2: After 2.1, check if the new second line is "Drucken Teilen"
            if len(lines) > 1 and lines[1].strip() == "Drucken Teilen":
                lines.pop(1)
                changes_count['www.fr.de']['rule_2_2'] += 1

        elif source_domain == 'www.focus.de':
            # Rule 3.1: Check first line for specific text
            if lines and lines[0].strip() == "Bitte markieren Sie die entsprechenden Wörter im Text. Mit nur zwei Klicks melden Sie den Fehler der Redaktion.":
                lines.pop(0)
                changes_count['www.focus.de']['rule_3_1'] += 1

            # Rule 3.2: Handle "E-Mail" first line case
            if lines and lines[0].strip() == "E-Mail":
                # Remove first 7 lines
                if len(lines) > 7:
                    lines = lines[7:]
                    # Check if the new first line is shorter than 100 characters
                    if lines and len(lines[0]) < 100:
                        lines.pop(0)
                    changes_count['www.focus.de']['rule_3_2'] += 1

            # Rule 3.3: Remove lines ending with specific newsletter text
            newsletter = "Hier können Sie den Newsletter ganz einfach und kostenlos abonnieren."
            lines = [line for line in lines if not line.strip().endswith(newsletter)]
            changes_count['www.focus.de']['rule_3_3'] += 1

        # Replace all \n with whitespace
        joined_text = ' '.join(lines)

        cleaned_text = re.sub(r'\(siehe Update vom[^)]*\)', '', joined_text)

        # Update the text column with the cleaned text
        df.at[index, 'text'] = cleaned_text

        # Check if the cleaned text is empty after processing, if so then add to removal list
        if not cleaned_text.strip():
            removed_indices.append(index)

    # Drop rows that have been marked for removal
    print(f"\nRemoving {removed_indices} rows with empty, None, or NaN text values")
    df_cleaned = df.drop(removed_indices)

    # Print summary of changes
    print("\nCleaning summary:")
    for domain, rules in changes_count.items():
        print(f"\n{domain}:")
        for rule, count in rules.items():
            print(f"  - {rule}: {count} changes")

    # Save the cleaned dataset
    df_cleaned.drop('original_text', axis=1, inplace=True)
    df_cleaned.to_csv(output_file_path, index=False)

    return df_cleaned

In [ ]:
output_file_path = "/content/drive/MyDrive/Colab Notebooks/t1an/cleaned_articles_dataset.csv"

# Run the cleaning process
cleaned_df = clean_dataset(data, output_file_path)

#print(cleaned_df[['source_domain', 'text']].head())

[] rows with initially empty/None/NaN text

Removing [] rows with empty, None, or NaN text values

Cleaning summary:

www.bild.de:
  - rule_1_1: 554 changes
  - rule_1_2: 575 changes

www.fr.de:
  - rule_2_1: 1480 changes
  - rule_2_2: 3248 changes

www.focus.de:
  - rule_3_1: 754 changes
  - rule_3_2: 201 changes
  - rule_3_3: 1849 changes

Symbol:
  - rule: 28440 changes

Reference:
  - short_slash_lines: 1699 changes

Time:
  - time: 9597 changes


In [ ]:
cleaned_df

,id,source_domain,url,title,text,label_3,label_5
0,887d66cad21d7daa12633e6aa18250a7a34ffa4913cf41...,www.mmnews.de,https://www.mmnews.de/politik/28379-bericht-sc...,Schulz: Gratisflüge mit EU-Jets für's Parteive...,Schulz: Gratisflüge mit EU-Jets für's Parteive...,1,center-right
1,7af76faf980452e667200c8e0427bdc5a241c5a9fe73c8...,www.bild.de,https://www.bild.de/politik/inland/politik-inl...,Sachsen hat gewählt: Der Krimi um die leeren A...,Irrer Krimi um die Zahl der AfD-Sitze im neuen...,1,center-right
2,444593843e2d0892f5ac204bcfcfafbaf72d82c4e565bf...,www.fr.de,https://www.fr.de/politik/boris-johnson-nicola...,SNP-Chefin greift Boris Johnson an - und forde...,SNP-Chefin greift Boris Johnson an - und forde...,0,center-left
3,4bc150c2bd195ee0edd66c419ca547d2696ae95891bf72...,www.mmnews.de,https://www.mmnews.de/politik/101548-siemens-c...,Siemens-Chef Kaeser will nach Saudi-Arabien,Siemens-Chef Kaeser will nach Saudi-Arabien An...,1,center-right
4,9572abfa63f0fec3fc9e9ed216474dcdd87e40447d239c...,www.focus.de,https://www.focus.de/politik/deutschland/theme...,"""Maybrit Illner"": Kanzleramtschef Altmaier ver...",Die von der SPD lange abgelehnte Große Koaliti...,1,center-right
...,...,...,...,...,...,...,...
13850,41d0aff870eb3a8c9609a6c474a0e6591dfe89a6500288...,www.n-tv.de,https://www.n-tv.de/politik/Papst-bedauert-Hag...,"""Es schmerzt mich sehr"": Papst bedauert Hagia-...","Die Entscheidung der Türkei, die Hagia Sophia ...",1,center
13851,40b4e05ee74eb43c09bba6cc69ed8ee0bde78a8d312210...,www.mmnews.de,https://www.mmnews.de/politik/145047-oesterrei...,Österreichs Kanzler: EU darf keine Schulden-Un...,Österreichs Kanzler: EU darf keine Schulden-Un...,1,center-right
13852,d9eec36ed123d5eb5b3b56ed466e97f4d8fb1b50710cd1...,www.bild.de,https://www.bild.de/politik/inland/bundestagsw...,Der Partei fehlen Themen - Panik-Pressekonfere...,"Die Flaute in Umfragen, der nach rechts außen ...",1,center-right
13853,196185390051889797bcbed15b632d32932e1a4c3250c3...,www.tichyseinblick.de,https://www.tichyseinblick.de/meinungen/spd-pa...,SPD-Parteitag: Andrea Nahles fügt sich den Rea...,Im Beisein der Vorsitzenden des Deutschen Gewe...,2,far-right


In [ ]:
cleaned_df.iloc[209]["text"]

'Verfassungsschutzchef Hans-Georg Maaßen wird nicht wie geplant als Sonderbeauftragter ins Innenministerium versetzt. Nach Tagesspiegel-Informationen geschieht das auf Maaßens eigenen Wunsch hin.\nAllerdings hat der 55-Jährige dem Vernehmen nach auch neuerlich für Verärgerung gesorgt. In seiner Abschiedsrede, deren Manuskript im Bundesamt für Verfassungsschutz verteilt wurde, habe er heftige Kritik an Teilen der Koalition geübt, vor allem an der SPD, und seine umstrittenen Äußerungen zu „Hetzjagden“ bei einer Demonstration in Chemnitz wieder massiv verteidigt. Auch habe er von teilweise linksradikalen Kräften bei den Sozialdemokraten gesprochen.\nBei einer Abschiedsrede vor europäischen Kollegen in Warschau am 18. Oktober soll Maaßen beklagt haben, seine Äußerungen zu den Vorfällen in Chemnitz seien für diese Kräfte willkommener Anlass gewesen, einen Bruch der großen Koalition zu provozieren. Das berichten der "Spiegel" und die Deutsche Presse-Agentur übereinstimmend. Maaßen sagte demn